<a href="https://colab.research.google.com/github/EshaShetty270607/agent_demo_loop/blob/main/Copy_of_agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [ ]:
# Quiet ensures your UI is not full of logs.
!pip install google-genai --quiet
print("Done.")

Done.


In [ ]:
# @title User input needed.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section."
        "Documents can be related to any topic - sports, manga, food, music etc."
        "Have a minimum of 4 sentences with each document."
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),
    "sourdough": (
        "Sourdough bread is baked in a Dutch oven preheated to 475F (245C). "
        "Bake it covered for 20 minutes to trap steam, then uncover and bake "
        "another 20 to 25 minutes until the crust is deep brown. The dough "
        "should ferment for 8 to 12 hours at room temperature with an active "
        "starter. Let the loaf cool for at least one hour before slicing."
    ),
    "steak": (
        "Pat a steak dry and season it generously with salt before cooking. "
        "Sear it in a very hot cast iron pan for 3 to 4 minutes per side for "
        "medium rare. Add butter, garlic and thyme in the last minute and "
        "baste the steak. Rest it for 5 to 10 minutes before slicing so the "
        "juices redistribute."
    ),
    "pasta": (
        "Bring a large pot of water to a rolling boil before adding pasta. "
        "Add about 1 tablespoon of salt per 4 quarts of water so the pasta is "
        "seasoned. Cook spaghetti for 9 to 11 minutes until al dente, and "
        "reserve a cup of starchy pasta water. Toss the pasta in the sauce "
        "with a splash of that water to help it cling."
    ),
    "roast_chicken": (
        "Roast a whole chicken at 425F (220C) for about 1 hour 15 minutes. "
        "Dry the skin well and rub it with butter and salt for crispy skin. "
        "The chicken is done when the thickest part of the thigh reaches "
        "165F (74C). Rest it for 15 minutes before carving."
    ),
}

In [ ]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check
print(search_documents("minimum of 4"))


[Source: sample_protocol]
Students must fill in documents in this section.Documents can be related to any topic - sports, manga, food, music etc.Have a minimum of 4 sentences with each document.Duplicate this for a minimum of 4 documents to test your agent.

[Source: roast_chicken]
Roast a whole chicken at 425F (220C) for about 1 hour 15 minutes. Dry the skin well and rub it with butter and salt for crispy skin. The chicken is done when the thickest part of the thigh reaches 165F (74C). Rest it for 15 minutes before carving.


In [ ]:
# @title Choose the model
MODEL_NAME = "gemini-3.1-flash-lite"  # if this errors, try "gemini-3-flash-preview"

SYSTEM_INSTRUCTION = '''
You are a chef training at a Michelin star restaurant.
When you need specific information about temperatures,
ingredients to use, cook time etc. use the search_documents tool.
Only answer based on what you find in the documents.
If the documents do not contain the answer, say so clearly.
'''
print("Model ready:", MODEL_NAME)

Model ready: gemini-3.1-flash-lite


In [ ]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'topics students share documents on'})
  Result preview: [Source: sample_protocol]
Students must fill in documents in this section.Documents can be related t...

Final answer:
According to the documents, students are to share documents related to any topic, such as sports, manga, food, or music. Each document must contain a minimum of four sentences.


In [ ]:
# @title Ask follow up questions
questions = [
    "At what temperature should I bake sourdough bread?",
    "How long should I rest a steak after searing it?",
    "How much salt goes in the pasta water?",
    "What internal temperature should roast chicken reach?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


Question: At what temperature should I bake sourdough bread?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'sourdough bread baking temperature'})
  Result preview: [Source: sourdough]
Sourdough bread is baked in a Dutch oven preheated to 475F (245C). Bake it cover...

Final answer:
To bake sourdough bread, you should use a Dutch oven preheated to 475°F (245°C).

Question: How long should I rest a steak after searing it?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'resting time for steak after searing'})
  Result preview: [Source: steak]
Pat a steak dry and season it generously with salt before cooking. Sear it in a very...

Final answer:
You should rest the steak for 5 to 10 minutes before slicing to allow the juices to redistribute.

Question: How much salt goes in the pasta water?
--------------------------------------------------
Step 1: model calls search_documents({'query': '

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)